<a href="https://colab.research.google.com/github/riccardogs/PyTorch_tutorial/blob/main/Optimizing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimizing Model Parameters

Now that we have a model and data it’s time to train, validate and test our model by optimizing its parameters on our data. Training a model is an iterative process; in each iteration the model makes a guess about the output, calculates the error in its guess (loss), collects the derivatives of the error with respect to its parameters (as we saw in the previous section), and optimizes these parameters using gradient descent. For a more detailed walkthrough of this process, check out this video on backpropagation from 3Blue1Brown (https://www.youtube.com/watch?v=tIeHLnjs5U8).

# Prerequisite Code

We load the code from the previous sections on Datasets & DataLoaders and Build Model.

In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

# Hyperparameters

Hyperparameters are adjustable parameters that let you control the model optimization process. Different hyperparameter values can impact model training and convergence rates (read more about hyperparameter tuning: https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)

We define the following hyperparameters for training:
- Number of Epochs - the number of times to iterate over the dataset
- Batch Size - the number of data samples propagated through the network before the parameters are updated
- Learning Rate - how much to update models parameters at each batch/epoch. Smaller values yield slow learning speed, while large values may result in unpredictable behavior during training.

In [5]:
learning_rate = 1e-3
# learning_rate = 1e-3: definisce il tasso di apprendimento (learning rate)
# - 1e-3 è la notazione scientifica per 0.001 (un millesimo)
# - Determina QUANTO il modello modifica i pesi ad ogni passo
# - Funziona come il "passo" con cui il modello scende lungo la curva della loss
#
# EFFETTO DEL LEARNING RATE:
# - Troppo ALTO (es. 1.0): il modello fa passi troppo grandi
#   * Rischia di "saltare" oltre il minimo, non converge
#   * La loss potrebbe divergere (andare a infinito)
#
# - Troppo BASSO (es. 1e-6): il modello fa passi minuscoli
#   * Converge molto lentamente
#   * Rischia di fermarsi in minimi locali
#   * Servono molte epoche
#
# - 1e-3 (0.001): è un valore di partenza comune che funziona bene in molti casi

batch_size = 64
# batch_size = 64: definisce quante immagini processare in parallelo
# - Ogni batch contiene 64 immagini
# - Il modello vede 64 immagini, calcola l'errore medio, poi aggiorna i pesi
#
# EFFETTO DEL BATCH SIZE:
# - Batch PICCOLO (es. 1-16):
#   * Aggiornamenti frequenti
#   * Più rumore nei gradienti (può aiutare a uscire da minimi locali)
#   * Richiede più tempo per epoca (più iterazioni)
#
# - Batch GRANDE (es. 256-1024):
#   * Gradienti più stabili (media su più campioni)
#   * Aggiornamenti meno frequenti
#   * Richiede più memoria GPU
#   * Può portare a minimi più "affilati" (generalizza peggio?)
#
# - 64: è un buon compromesso, abbastanza piccolo per essere gestibile,
#   abbastanza grande per gradienti stabili

epochs = 5
# epochs = 5: definisce il numero di epoche di addestramento
# - Un'epoca = un PASSAGGIO COMPLETO su TUTTO il dataset di training
# - Con 60.000 immagini e batch_size=64:
#   * 60.000 / 64 = 938 batch per epoca
#   * 5 epoche = 5 × 938 = 4.690 aggiornamenti dei pesi
#
# COSA SUCCEDE IN UN'EPOCA:
# 1. Il modello vede tutte le 60.000 immagini (in ordine casuale)
# 2. Per ogni batch, calcola loss e aggiorna i pesi
# 3. Alla fine, potrebbe aver imparato qualcosa
#
# TROPPE POCHE EPOCHE (es. 1-2):
# - Il modello non ha abbastanza tempo per imparare (underfitting)
#
# TROPPE EPOCHE (es. 1000):
# - Il modello potrebbe imparare a memoria i dati di training (overfitting)
# - Spreco di tempo e risorse
#
# 5 epoche: per Fashion-MNIST è un numero piccolo (per vedere progressi veloci)
# Di solito si usano 10-50 epoche per questo dataset

# COME INTERAGISCONO:
# - In ogni epoca: il modello vede TUTTI i dati
# - In ogni epoca: ci sono (dataset_size / batch_size) iterazioni
# - A ogni iterazione: i pesi vengono aggiornati di (learning_rate × gradiente)

# CALCOLO DEL NUMERO DI AGGIORNAMENTI:
# dataset_size = 60.000 (Fashion-MNIST training)
# batch_size = 64
# iterazioni_per_epoca = 60.000 / 64 ≈ 938
# epoche = 5
# aggiornamenti_totali = 938 × 5 = 4.690

# FORMULA DI AGGIORNAMENTO PESI:
# nuovi_pesi = vecchi_pesi - learning_rate × gradienti

# Optimization Loop

Once we set our hyperparameters, we can then train and optimize our model with an optimization loop. Each iteration of the optimization loop is called an epoch.

Each epoch consists of two main parts:
- The Train Loop - iterate over the training dataset and try to converge to optimal parameters.
- The Validation/Test Loop - iterate over the test dataset to check if model performance is improving.

Let’s briefly familiarize ourselves with some of the concepts used in the training loop. Jump ahead to see the Full Implementation of the optimization loop.


# Loss Function

When presented with some training data, our untrained network is likely not to give the correct answer. Loss function measures the degree of dissimilarity of obtained result to the target value, and it is the loss function that we want to minimize during training. To calculate the loss we make a prediction using the inputs of our given data sample and compare it against the true data label value.

Common loss functions include nn.MSELoss (Mean Square Error) for regression tasks, and nn.NLLLoss (Negative Log Likelihood) for classification. nn.CrossEntropyLoss combines nn.LogSoftmax and nn.NLLLoss.

We pass our model’s output logits to nn.CrossEntropyLoss, which will normalize the logits and compute the prediction error.

In [6]:
# Initialize the loss function - inizializza la funzione di perdita (loss function)

loss_fn = nn.CrossEntropyLoss()
# loss_fn = nn.CrossEntropyLoss(): crea un oggetto che calcola la Cross Entropy Loss
# - nn.CrossEntropyLoss è una classe di PyTorch per il calcolo della loss
# - È la funzione di perdita PIÙ COMUNE per problemi di CLASSIFICAZIONE MULTICLASSE
# - Come Fashion-MNIST che ha 10 classi (T-shirt, Trouser, ecc.)

# COSA FA CROSSENTROPYLOSS:
# Combina DUE operazioni in una, per stabilità numerica:
# 1. Softmax: converte i logits in probabilità
# 2. Negative Log Likelihood: calcola la perdita

# FORMULA CONCETTUALE:
# loss = -log(probabilità_della_classe_corretta)
#
# Se il modello è molto sicuro e indovina:
#   probabilità = 0.95 → loss = -log(0.95) ≈ 0.05 (bassa)
#
# Se il modello è insicuro o sbaglia:
#   probabilità = 0.10 → loss = -log(0.10) ≈ 2.30 (alta)

# INPUT ATTESI DA CrossEntropyLoss:
# - PREDIZIONI (logits): tensor di forma (batch_size, num_classes)
#   * Esempio: (64, 10) per 64 immagini, 10 classi
#   * Sono i valori grezzi DAL MODELLO (NON applicare softmax prima!)
#
# - TARGET (etichette): tensor di forma (batch_size,) con interi da 0 a 9
#   * Esempio: [3, 5, 1, 0, 2, ...] per ogni immagine nel batch
#   * NON in formato one-hot! Accetta direttamente gli indici delle classi

# ESEMPIO CONCRETO:
# Supponiamo un batch di 2 immagini:
# - La prima immagine è un Dress (classe 3)
# - La seconda è una Sneaker (classe 7)

# Il modello produce logits:
#logits = torch.tensor([[2.1, -1.3, 0.5, 3.2, -0.8, 1.7, -2.1, 0.3, -0.5, 1.2],  # prima immagine
#                       [1.2, -0.3, 2.1, -1.5, 0.7, -2.3, 1.8, 3.5, -1.1, 0.9]]) # seconda immagine

# Le etichette vere:
#targets = torch.tensor([3, 7])  # prima è classe 3, seconda è classe 7

# Calcolo loss:
#loss = loss_fn(logits, targets)
#print(loss)  # Un singolo numero (lo scalare di loss)

# COSA SUCCEDE INTERNAMENTE:
# 1. Per la prima immagine: softmax sui 10 logits → probabilità
# 2. Prende la probabilità della classe corretta (classe 3)
# 3. Calcola -log(probabilità)
# 4. Stessa cosa per la seconda immagine
# 5. Fa la MEDIA delle loss del batch (di default)

# PERCHÉ USARE QUESTA LOSS:
# - È specifica per classificazione
# - Penalizza molto gli errori sicuri (quando il modello è convinto ma sbaglia)
# - Incoraggia il modello ad aumentare la probabilità della classe corretta
# - La combinazione con softmax è numericamente stabile

# NOTA: Non applichiamo softmax nel modello!
# Il modello produce logits, e CrossEntropyLoss si aspetta logits
# Se applicassi softmax prima, farei softmax due volte

# Optimizer

Optimization is the process of adjusting model parameters to reduce model error in each training step. Optimization algorithms define how this process is performed (in this example we use Stochastic Gradient Descent). All optimization logic is encapsulated in the optimizer object. Here, we use the SGD optimizer; additionally, there are many different optimizers available in PyTorch such as ADAM and RMSProp, that work better for different kinds of models and data.

We initialize the optimizer by registering the model’s parameters that need to be trained, and passing in the learning rate hyperparameter.

Inside the training loop, optimization happens in three steps:
- Call optimizer.zero_grad() to reset the gradients of model parameters. - Gradients by default add up; to prevent double-counting, we explicitly zero them at each iteration.
- Backpropagate the prediction loss with a call to loss.backward().

PyTorch deposits the gradients of the loss w.r.t. each parameter.
Once we have our gradients, we call optimizer.step() to adjust the parameters by the gradients collected in the backward pass.

In [7]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# optimizer = torch.optim.SGD(...): crea un ottimizzatore che aggiorna i pesi del modello

# ANALIZZIAMO I COMPONENTI:

# 1. torch.optim.SGD
#    - torch.optim: modulo di PyTorch con algoritmi di ottimizzazione
#    - SGD: Stochastic Gradient Descent (Discesa del Gradiente Stocastica)
#    - È l'algoritmo base per aggiornare i pesi

# 2. model.parameters()
#    - model.parameters(): metodo che restituisce TUTTI i parametri addestrabili del modello
#    - Include pesi e bias di TUTTI i layer (Linear, ecc.)
#    - Sono i tensor che hanno requires_grad=True
#    - L'ottimizzatore terrà traccia di questi parametri e li aggiornerà

# 3. lr=learning_rate
#    - lr: learning rate (tasso di apprendimento)
#    - learning_rate: la variabile che abbiamo definito prima (1e-3 = 0.001)
#    - Determina l'ampiezza del passo di aggiornamento

# COME FUNZIONA SGD (Stochastic Gradient Descent):
# Formula di aggiornamento:
# nuovo_peso = vecchio_peso - lr * gradiente
#
# Dove:
# - vecchio_peso: valore attuale del parametro
# - lr: learning rate (controlla quanto modificare)
# - gradiente: quanto quel peso contribuisce all'errore (calcolato da backward())

# ESEMPIO CONCRETO:
# Supponiamo un peso w = 2.5, e il suo gradiente = 0.8
# Con lr = 0.001:
# nuovo_w = 2.5 - 0.001 * 0.8 = 2.5 - 0.0008 = 2.4992



# CICLO TIPICO DI ADDESTRAMENTO:

# for epoch in range(epochs):
#    for batch in dataloader:
#        # 1. Forward pass: calcola predizioni
#        pred = model(images)

#        # 2. Calcola loss
#        loss = loss_fn(pred, labels)

#        # 3. Azzera gradienti (importante!)
#        optimizer.zero_grad()

#        # 4. Backward pass: calcola gradienti
#        loss.backward()

#        # 5. Aggiorna pesi usando i gradienti
#        optimizer.step()
        # Questo è dove SGD fa: parametro -= lr * parametro.grad


# PERCHÉ optimizer.zero_grad() È IMPORTANTE:
# - I gradienti si accumulano a ogni .backward()
# - Se non azzeri, i gradienti dei batch precedenti si sommano
# - optimizer.zero_grad() azzera TUTTI i gradienti dei parametri

# VARIANTI DI OTTIMIZZATORI (non solo SGD):
# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # Adam (più popolare)
# optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)  # RMSprop
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)  # Adam con weight decay

# PERCHÉ SGD:
# - È l'algoritmo più semplice e fondamentale
# - Funziona bene per molti problemi
# - Buon punto di partenza per imparare
# - Altri ottimizzatori (Adam) sono variazioni più sofisticate

# COSA CONTIENE optimizer:
# - Riferimenti a tutti i parametri del modello
# - Learning rate corrente
# - Stato interno (per ottimizzatori come Adam, che hanno momento)
# - Metodi come .step(), .zero_grad(), .state_dict()

# Full Implementation

We define train_loop that loops over our optimization code, and test_loop that evaluates the model’s performance against our test data.

In [8]:
def train_loop(dataloader, model, loss_fn, optimizer):
    # train_loop: funzione che esegue un'epoca di addestramento
    # Parametri:
    # - dataloader: contiene i batch di training
    # - model: la rete neurale da addestrare
    # - loss_fn: funzione di loss (es. CrossEntropyLoss)
    # - optimizer: algoritmo di ottimizzazione (es. SGD)

    size = len(dataloader.dataset)
    # size: numero TOTALE di campioni nel dataset (es. 60000 per Fashion-MNIST training)

    # Set the model to training mode
    model.train()
    # model.train(): imposta il modello in MODALITÀ TRAINING
    # - Importante per layer come Dropout e BatchNorm che si comportano diversamente in training
    # - In training: Dropout attivo, BatchNorm usa statistiche del batch
    # - Anche se qui non abbiamo questi layer, è buona pratica includerlo

    for batch, (X, y) in enumerate(dataloader):
        # for batch, (X, y) in enumerate(dataloader): itera su tutti i batch
        # - batch: indice del batch (0, 1, 2, ...)
        # - X: tensor di immagini del batch corrente (es. 64, 1, 28, 28)
        # - y: tensor di etichette del batch corrente (es. 64,)

        # Compute prediction and loss
        pred = model(X)
        # pred = model(X): forward pass, calcola predizioni per il batch
        # - pred ha forma (batch_size, num_classes) es. (64, 10)

        loss = loss_fn(pred, y)
        # loss = loss_fn(pred, y): calcola la loss tra predizioni e verità
        # - loss è un tensor scalare (un numero)

        # Backpropagation
        loss.backward()
        # loss.backward(): calcola i gradienti per tutti i parametri
        # I gradienti vengono accumulati in .grad di ogni parametro

        optimizer.step()
        # optimizer.step(): aggiorna i pesi usando i gradienti
        # Formula: peso -= learning_rate * peso.grad

        optimizer.zero_grad()
        # optimizer.zero_grad(): azzera i gradienti per la prossima iterazione
        # Se non lo facessi, i gradienti si sommerebbero a ogni batch

        if batch % 100 == 0:
            # if batch % 100 == 0: ogni 100 batch, stampa progresso
            loss, current = loss.item(), batch * batch_size + len(X)
            # loss.item(): estrae il valore numerico della loss (da tensor a float)
            # current: numero di campioni processati finora
            #   batch * batch_size: campioni dei batch completi
            #   + len(X): campioni del batch corrente (ultimo batch potrebbe essere più piccolo)

            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
            # Stampa loss e progresso (es. loss: 2.345  [  128/60000])


def test_loop(dataloader, model, loss_fn):
    # test_loop: funzione che valuta il modello sul set di test
    # - Non aggiorna i pesi, solo calcola accuratezza e loss media

    model.eval()
    # model.eval(): imposta il modello in MODALITÀ VALUTAZIONE
    # - Disattiva Dropout
    # - BatchNorm usa statistiche accumulate (non del batch)
    # - Fondamentale per ottenere valutazioni corrette

    size = len(dataloader.dataset)
    # size: numero totale di campioni di test (es. 10000)

    num_batches = len(dataloader)
    # num_batches: numero di batch (es. 10000/64 ≈ 157)

    test_loss, correct = 0, 0
    # test_loss: accumulatore per sommare le loss di tutti i batch
    # correct: accumulatore per contare le predizioni corrette

    # Evaluating the model with torch.no_grad()
    with torch.no_grad():
        # with torch.no_grad(): disabilita il calcolo dei gradienti
        # - Risparmia memoria e calcoli (non serve backward in test)
        # - I tensor creati dentro non richiedono gradienti

        for X, y in dataloader:
            # Itera su tutti i batch di test

            pred = model(X)
            # pred: predizioni del modello, forma (batch_size, 10)

            test_loss += loss_fn(pred, y).item()
            # test_loss += ...: somma la loss di questo batch
            # .item() converte da tensor a float

            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            # (pred.argmax(1) == y): confronta classe predetta con vera
            #   - pred.argmax(1): indice della classe con probabilità massima per ogni campione
            #   - == y: confronto booleano (True se indovinato)
            # .type(torch.float): converte True/False in 1.0/0.0
            # .sum(): conta quanti veri (quante predizioni corrette)
            # .item(): estrae il numero

    test_loss /= num_batches
    # test_loss /= num_batches: calcola la loss MEDIA (dividendo per numero di batch)

    correct /= size
    # correct /= size: calcola l'accuratezza (corretti / totale)

    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    # Stampa:
    # - Accuratezza in percentuale (es. 85.3%)
    # - Loss media sul test set